# Figure 2

Draws the drought-response maps, class summaries, and regional storage reconstruction.


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap, LogNorm
from matplotlib.patches import Patch, Rectangle
from pyproj import Geod, Transformer
from scipy.interpolate import PchipInterpolator
from scipy.ndimage import generic_filter
from shapely.geometry import LineString


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/RECON_MAIN_2011_2023 from the current working directory.')


def display_path(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(ROOT))
    except ValueError:
        return str(path)


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
CLUSTER_LABELS_CSV = RECON / 'metrics' / 'clustering' / 'cluster_labels.csv'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'Fig2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MRVA_BOUNDARY_PATH = ROOT / 'assets' / 'spatial' / 'mrva_boundary.geojson'
MISSISSIPPI_RIVER_GMT_PATH = ROOT / 'assets' / 'spatial' / 'mississippi_river.gmt'

GRID_CRS = 'EPSG:5070'
LONLAT_CRS = 'EPSG:4326'
EXPORT_DPI = 600
MAP_DISPLAY_MEDIAN_FILTER_SIZE = 3
MAP_BOUNDARY_COLOR = '#1f1f1f'
MAP_BOUNDARY_LW = 0.45
MAP_RIVER_COLOR = '#B7DDE8'
MAP_RIVER_LW = 0.42
BOX_FRAME_LINEWIDTH = 0.75
BOX_XTICK_LABELSIZE = 13
BOX_YTICK_LABELSIZE = 11
SCALEBAR_GEOD = Geod(ellps='WGS84')

CLASS_ORDER = ['Fast recovery', 'Slow recovery', 'Buffered']
CLASS_COLORS = {
    'Fast recovery': '#2D5FB8',
    'Slow recovery': '#C44E72',
    'Buffered': '#13A8A2',
}

DECLINE_CMAP = LinearSegmentedColormap.from_list('fig2_decline', ['#F7F7F7', '#F2B07D', '#B64342'], N=256)
RDOWN_CMAP = LinearSegmentedColormap.from_list('fig3_rdown', ['#F7F4EA', '#E7B06A', '#9F493D'], N=256)
RECOVERY_CMAP = LinearSegmentedColormap.from_list('fig3_recovery', ['#F2F6EF', '#8FC9BE', '#2F6F8F'], N=256)
MEMORY_CMAP = LinearSegmentedColormap.from_list('fig3_memory', ['#F5F1F5', '#B99ABB', '#6E496E'], N=256)
LONG_RECOVERY_CMAP = LinearSegmentedColormap.from_list('fig3_long_recovery', ['#F1F7EC', '#A8CFA6', '#2E7D4F'], N=256)

METRIC_SPECS = [
    {
        'column': 'decline_m',
        'stem': 'Fig2_decline_2012',
        'label': '2012 drought WTD decline (m)',
        'cmap': DECLINE_CMAP,
        'vmin': 0.0,
        'vmax': 4.0,
        'ticks': [0, 1, 2, 3, 4],
        'extend': 'both',
        'scale_bar_km': 100,
    },
    {
        'column': 'Rdown_m_per_month',
        'stem': 'Fig2_Rdown',
        'label': 'Deepening rate (m month$^{-1}$)',
        'cmap': RDOWN_CMAP,
        'vmin': 0.0,
        'vmax': 0.8,
        'ticks': [0.0, 0.2, 0.4, 0.6, 0.8],
        'extend': 'both',
    },
    {
        'column': 'RR_early',
        'stem': 'Fig2_RR_early',
        'label': 'Early recovery fraction',
        'cmap': RECOVERY_CMAP,
        'vmin': 0.0,
        'vmax': 1.0,
        'ticks': [0.0, 0.5, 1.0],
        'extend': 'both',
    },
    {
        'column': 'T50_months',
        'stem': 'Fig2_T50',
        'label': 'Time to 50% recovery (months)',
        'cmap': MEMORY_CMAP,
        'vmin': 1.0,
        'vmax': 72.0,
        'ticks': [1, 3, 6, 12, 24, 72],
        'ticklabels': ['1', '3', '6', '12', '24', '72'],
        'norm': 'log',
        'extend': 'both',
    },
    {
        'column': 'RR2019',
        'stem': 'Fig2_RR2019',
        'label': 'Long-term recovery fraction by 2019',
        'cmap': LONG_RECOVERY_CMAP,
        'vmin': 0.0,
        'vmax': 3.0,
        'ticks': [0, 1, 2, 3],
        'extend': 'both',
        'missing_color': '#F1F7EC',
    },
]

BOXPLOT_SPECS = [
    {
        'column': 'decline_m',
        'stem': 'Fig2_box_decline',
        'label': '2012 drought WTD decline (m)',
        'unit_label': 'm',
        'yscale': 'linear',
    },
    {
        'column': 'Rdown_m_per_month',
        'stem': 'Fig2_box_Rdown',
        'label': 'Deepening rate (m month$^{-1}$)',
        'unit_label': 'm / month',
        'yscale': 'linear',
        'ylim': (0, 1),
    },
    {
        'column': 'RR_early',
        'stem': 'Fig2_box_RR_early',
        'label': 'Early recovery fraction',
        'unit_label': '(-)',
        'yscale': 'linear',
        'ylim': (0, 2),
        'yticks': [0, 0.5, 1, 1.5, 2],
        'reference_lines': [0, 1],
    },
    {
        'column': 'T50_months',
        'stem': 'Fig2_box_T50',
        'label': 'Time to 50% recovery (months)',
        'unit_label': 'month',
        'yscale': 'log',
        'yticks': [1, 3, 12, 72],
    },
    {
        'column': 'RR2019',
        'stem': 'Fig2_box_RR2019',
        'label': 'Long-term recovery fraction by 2019',
        'unit_label': '(-)',
        'yscale': 'linear',
        'ylim': (0, 5),
        'yticks': [0, 1, 2, 3, 4, 5],
        'reference_lines': [0, 1],
    },
]

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
    'axes.linewidth': 0.65,
    'legend.frameon': False,
})

print('Response metrics:', display_path(CLUSTER_LABELS_CSV))
print('Output:', display_path(OUT_DIR))


In [ ]:
labels = pd.read_csv(CLUSTER_LABELS_CSV)
required_columns = list(dict.fromkeys(
    ['row', 'col', 'x', 'y', 'valid_for_clustering', 'response_class']
    + [spec['column'] for spec in METRIC_SPECS]
    + [spec['column'] for spec in BOXPLOT_SPECS]
))
missing = [col for col in required_columns if col not in labels.columns]
if missing:
    raise KeyError(f'Missing columns in cluster labels: {missing}')

labels['valid_for_clustering'] = labels['valid_for_clustering'].astype(bool)
n_rows = int(labels['row'].max()) + 1
n_cols = int(labels['col'].max()) + 1
rows = labels['row'].to_numpy(dtype=int)
cols = labels['col'].to_numpy(dtype=int)
valid_mask = labels['valid_for_clustering'].to_numpy(dtype=bool)


def grid_edge_lonlat() -> tuple[np.ndarray, np.ndarray]:
    x_by_col = labels.groupby('col')['x'].first().sort_index().to_numpy(dtype=np.float64)
    y_by_row = labels.groupby('row')['y'].first().sort_index().to_numpy(dtype=np.float64)
    dx = float(np.nanmedian(np.diff(x_by_col)))
    dy = float(np.nanmedian(np.diff(y_by_row)))
    x_edges = np.concatenate([[x_by_col[0] - 0.5 * dx], x_by_col + 0.5 * dx])
    y_edges = np.concatenate([[y_by_row[0] - 0.5 * dy], y_by_row + 0.5 * dy])
    xx, yy = np.meshgrid(x_edges, y_edges)
    transformer = Transformer.from_crs(GRID_CRS, LONLAT_CRS, always_xy=True)
    lon, lat = transformer.transform(xx, yy)
    return np.asarray(lon), np.asarray(lat)


def read_gmt_segments(path: Path) -> list[np.ndarray]:
    if not path.exists():
        return []
    segments = []
    current = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                segments.append(np.asarray(current, dtype=np.float64))
                current = []
            continue
        lon, lat = line.split()[:2]
        current.append((float(lon), float(lat)))
    if current:
        segments.append(np.asarray(current, dtype=np.float64))
    return segments


def plot_line_geometry_lonlat(ax, geometry, **kwargs) -> None:
    if geometry is None or geometry.is_empty:
        return
    geom_type = geometry.geom_type
    if geom_type == 'LineString':
        x, y = geometry.xy
        ax.plot(np.asarray(x), np.asarray(y), **kwargs)
    elif geom_type in {'MultiLineString', 'GeometryCollection'}:
        for part in geometry.geoms:
            plot_line_geometry_lonlat(ax, part, **kwargs)
    elif geom_type == 'Polygon':
        plot_line_geometry_lonlat(ax, geometry.boundary, **kwargs)
    elif geom_type == 'MultiPolygon':
        for part in geometry.geoms:
            plot_line_geometry_lonlat(ax, part.boundary, **kwargs)


def load_mrva_boundary_polygon():
    boundary = gpd.read_file(MRVA_BOUNDARY_PATH).to_crs(LONLAT_CRS)
    if hasattr(boundary.geometry, 'union_all'):
        return boundary.geometry.union_all()
    return boundary.unary_union


def clipped_segment_to_mrva(segment: np.ndarray):
    if len(segment) < 2:
        return None
    return LineString(segment).intersection(MRVA_POLYGON_GEOMETRY)


def rasterize_values(values: np.ndarray) -> np.ndarray:
    raster = np.full((n_rows, n_cols), np.nan, dtype=np.float32)
    raster[rows, cols] = values.astype(np.float32)
    return raster


def rasterize_metric(column: str) -> np.ndarray:
    values = labels[column].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
    values[~valid_mask] = np.nan
    return rasterize_values(values)


def nanmedian_quiet(window: np.ndarray) -> float:
    finite = window[np.isfinite(window)]
    if finite.size == 0:
        return np.nan
    return float(np.median(finite))


def smooth_display_raster(raster: np.ndarray, filter_size: int = MAP_DISPLAY_MEDIAN_FILTER_SIZE) -> np.ndarray:
    if filter_size <= 1:
        return raster
    display = generic_filter(raster, nanmedian_quiet, size=filter_size, mode='nearest')
    display[~np.isfinite(raster)] = np.nan
    return display.astype(np.float32)


MAP_LON_EDGES, MAP_LAT_EDGES = grid_edge_lonlat()
MAP_EXTENT_LONLAT = (
    float(np.nanmin(MAP_LON_EDGES)),
    float(np.nanmax(MAP_LON_EDGES)),
    float(np.nanmin(MAP_LAT_EDGES)),
    float(np.nanmax(MAP_LAT_EDGES)),
)
MRVA_POLYGON_GEOMETRY = load_mrva_boundary_polygon()
MRVA_BOUNDARY_GEOMETRY = MRVA_POLYGON_GEOMETRY.boundary
MISSISSIPPI_SEGMENTS_LONLAT = read_gmt_segments(MISSISSIPPI_RIVER_GMT_PATH)

print(f'Total MRVA cells: {len(labels):,}')
print(f'Valid classified cells: {valid_mask.sum():,}')


In [ ]:
def format_map_axis(ax: plt.Axes) -> None:
    ax.set_xlim(MAP_EXTENT_LONLAT[0], MAP_EXTENT_LONLAT[1])
    ax.set_ylim(MAP_EXTENT_LONLAT[2], MAP_EXTENT_LONLAT[3])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(axis='both', which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    mean_lat = 0.5 * (MAP_EXTENT_LONLAT[2] + MAP_EXTENT_LONLAT[3])
    ax.set_aspect(1.0 / np.cos(np.deg2rad(mean_lat)))


def draw_map_context(ax: plt.Axes) -> None:
    for segment in MISSISSIPPI_SEGMENTS_LONLAT:
        clipped = clipped_segment_to_mrva(segment)
        plot_line_geometry_lonlat(ax, clipped, color=MAP_RIVER_COLOR, lw=MAP_RIVER_LW, alpha=0.95, zorder=3)
    plot_line_geometry_lonlat(ax, MRVA_BOUNDARY_GEOMETRY, color=MAP_BOUNDARY_COLOR, lw=MAP_BOUNDARY_LW, zorder=4)


def draw_scale_bar(ax: plt.Axes, length_km: float = 100, anchor: tuple[float, float] = (0.60, 0.055)) -> None:
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    lon0 = xmin + anchor[0] * (xmax - xmin)
    lat0 = ymin + anchor[1] * (ymax - ymin)
    lon1, _, _ = SCALEBAR_GEOD.fwd(lon0, lat0, 90, length_km * 1000)
    bar_width = lon1 - lon0
    bar_height = 0.010 * (ymax - ymin)
    segment_width = bar_width / 2
    for i, facecolor in enumerate(['#111111', '#FFFFFF']):
        rect = Rectangle(
            (lon0 + i * segment_width, lat0),
            segment_width,
            bar_height,
            facecolor=facecolor,
            edgecolor='#111111',
            linewidth=0.45,
            clip_on=False,
            zorder=8,
        )
        ax.add_patch(rect)
    ax.text(
        lon0 + 0.5 * bar_width,
        lat0 + 2.0 * bar_height,
        f'{int(length_km)} km',
        ha='center',
        va='bottom',
        fontsize=7.5,
        fontweight='bold',
        fontfamily='Arial',
        color='#111111',
        clip_on=False,
        path_effects=[pe.withStroke(linewidth=1.6, foreground='white')],
        zorder=9,
    )


def plot_metric_map(spec: dict) -> Path:
    raw_values = labels[spec['column']].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
    raster = rasterize_metric(spec['column'])
    display_raster = smooth_display_raster(raster)
    cmap = spec['cmap'].copy()
    cmap.set_bad('#FFFFFF')

    color_scale = {'vmin': spec['vmin'], 'vmax': spec['vmax']}
    if spec.get('norm') == 'log':
        color_scale = {'norm': LogNorm(vmin=spec['vmin'], vmax=spec['vmax'])}

    fig, ax = plt.subplots(figsize=(3.35, 5.8), dpi=EXPORT_DPI)
    image = ax.pcolormesh(
        MAP_LON_EDGES,
        MAP_LAT_EDGES,
        display_raster,
        cmap=cmap,
        **color_scale,
        shading='flat',
        rasterized=True,
    )
    if 'missing_color' in spec:
        missing_values = np.full(len(labels), np.nan, dtype=np.float32)
        missing_values[valid_mask & ~np.isfinite(raw_values)] = 1.0
        missing_raster = rasterize_values(missing_values)
        missing_cmap = ListedColormap([spec['missing_color']])
        missing_cmap.set_bad((1.0, 1.0, 1.0, 0.0))
        ax.pcolormesh(
            MAP_LON_EDGES,
            MAP_LAT_EDGES,
            missing_raster,
            cmap=missing_cmap,
            vmin=0.5,
            vmax=1.5,
            shading='flat',
            rasterized=True,
            zorder=2,
        )

    draw_map_context(ax)
    format_map_axis(ax)
    if 'scale_bar_km' in spec:
        draw_scale_bar(ax, length_km=spec['scale_bar_km'])

    cbar = fig.colorbar(image, ax=ax, fraction=0.040, pad=0.025, extend=spec['extend'])
    cbar.set_ticks(spec['ticks'])
    if 'ticklabels' in spec:
        cbar.set_ticklabels(spec['ticklabels'])
    cbar.ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
    cbar.ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    cbar.set_label(spec['label'], rotation=90, labelpad=7)
    cbar.ax.tick_params(length=2.8, width=0.65, labelsize=10.0)
    cbar.outline.set_linewidth(0.6)

    out = OUT_DIR / f"{spec['stem']}.png"
    fig.savefig(out, dpi=EXPORT_DPI)
    plt.close(fig)
    return out


def plot_class_map() -> Path:
    class_to_index = {name: idx for idx, name in enumerate(CLASS_ORDER)}
    class_values = np.full(len(labels), np.nan, dtype=np.float32)
    for class_name, idx in class_to_index.items():
        class_values[(labels['response_class'] == class_name).to_numpy() & valid_mask] = idx
    class_raster = rasterize_values(class_values)

    cmap = ListedColormap([CLASS_COLORS[name] for name in CLASS_ORDER])
    cmap.set_bad('#FFFFFF')
    norm = BoundaryNorm(np.arange(-0.5, len(CLASS_ORDER) + 0.5, 1), cmap.N)

    fig, ax = plt.subplots(figsize=(4.35, 5.8), dpi=EXPORT_DPI)
    ax.pcolormesh(
        MAP_LON_EDGES,
        MAP_LAT_EDGES,
        class_raster,
        cmap=cmap,
        norm=norm,
        shading='flat',
        rasterized=True,
    )
    draw_map_context(ax)
    format_map_axis(ax)

    handles = [Patch(facecolor=CLASS_COLORS[name], edgecolor='none', label=name) for name in CLASS_ORDER]
    ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5), handlelength=1.0, handletextpad=0.4)

    out = OUT_DIR / 'Fig2_response_classes.png'
    fig.savefig(out, dpi=EXPORT_DPI)
    plt.close(fig)
    return out


CLASS_PLOT_LABELS = {
    'Fast recovery': 'F',
    'Slow recovery': 'S',
    'Buffered': 'B',
}


def apply_boxplot_axis_scale(ax: plt.Axes, spec: dict) -> None:
    if spec.get('yscale') == 'log':
        ax.set_yscale('log')
        ax.set_yticks(spec['yticks'])
        ax.set_yticklabels([str(tick) for tick in spec['yticks']])
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    elif spec.get('yscale') == 'symlog':
        ax.set_yscale('symlog', linthresh=spec.get('linthresh', 1.0))
        ax.set_yticks(spec['yticks'])
        ax.set_yticklabels([str(tick) for tick in spec['yticks']])
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    if 'ylim' in spec:
        ax.set_ylim(*spec['ylim'])
    if 'yticks' in spec and spec.get('yscale') == 'linear':
        ax.set_yticks(spec['yticks'])


def plot_metric_boxplot(spec: dict) -> Path:
    class_data = []
    positions = np.arange(1, len(CLASS_ORDER) + 1)
    for class_name in CLASS_ORDER:
        values = labels.loc[
            valid_mask & (labels['response_class'] == class_name), spec['column']
        ].replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=np.float64)
        class_data.append(values)

    fig, ax = plt.subplots(figsize=(5.0, 2.65), dpi=EXPORT_DPI)
    bp = ax.boxplot(
        class_data,
        positions=positions,
        widths=0.56,
        patch_artist=True,
        whis=(5, 95),
        showfliers=False,
        medianprops={'color': '#111111', 'linewidth': 1.15},
        whiskerprops={'color': '#4a4a4a', 'linewidth': 0.75},
        capprops={'color': '#4a4a4a', 'linewidth': 0.75},
    )
    for patch, class_name in zip(bp['boxes'], CLASS_ORDER):
        patch.set_facecolor(CLASS_COLORS[class_name])
        patch.set_alpha(0.72)
        patch.set_edgecolor('#333333')
        patch.set_linewidth(0.75)

    for y_ref in spec.get('reference_lines', []):
        ax.axhline(y_ref, color='#666666', lw=0.55, ls='--', alpha=0.55, zorder=0)
    apply_boxplot_axis_scale(ax, spec)
    ax.set_xlim(0.45, len(CLASS_ORDER) + 0.55)
    ax.set_xticks(positions)
    ax.set_xticklabels(
        [CLASS_PLOT_LABELS[name] for name in CLASS_ORDER],
        fontfamily='Arial',
        fontweight='bold',
        fontsize=BOX_XTICK_LABELSIZE,
    )
    ax.grid(axis='y', color='#E6E2DA', lw=0.55, alpha=0.75)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(BOX_FRAME_LINEWIDTH)
        spine.set_color('#111111')
    ax.set_ylabel(spec.get('unit_label', ''), fontsize=BOX_YTICK_LABELSIZE, labelpad=4)
    ax.tick_params(axis='x', length=0, pad=3)
    ax.tick_params(axis='y', length=2.8, width=BOX_FRAME_LINEWIDTH, labelsize=BOX_YTICK_LABELSIZE)

    out = OUT_DIR / f"{spec['stem']}.png"
    fig.savefig(out, dpi=EXPORT_DPI)
    plt.close(fig)
    return out


In [ ]:
GRACE_RECON_MONTHLY_PATH = (
    RECON / 'diagnostics' / 'regional_mass_correction' / 'regional_mass_correction_monthly.csv'
)
PANEL_GRID_COLOR = '#D6D6D6'
PANEL_SPINE_COLOR = '#2A2A2A'


def compute_grace_reconstruction_series() -> pd.DataFrame:
    monthly = pd.read_csv(GRACE_RECON_MONTHLY_PATH)
    return pd.DataFrame({
        'date': pd.to_datetime(monthly['month_label']),
        'grace_lwe_anom_cm': monthly['grace_anom_cm'],
        'reconstruction_storage_proxy_anom_cm': monthly['corrected_storage_proxy_cm'],
        'has_grace_solution': monthly['has_valid_grace'].astype(bool),
    })


def style_timeseries_axis(ax: plt.Axes) -> None:
    ax.set_axisbelow(True)
    ax.grid(axis='both', color=PANEL_GRID_COLOR, lw=0.45, alpha=0.62, zorder=0)
    for side in ['top', 'right', 'bottom', 'left']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color(PANEL_SPINE_COLOR)
        ax.spines[side].set_linewidth(0.72)
    ax.tick_params(
        axis='x', top=True, bottom=True, labeltop=False, labelbottom=True,
        direction='out', length=3.0, width=0.65, pad=2.0,
    )
    ax.tick_params(
        axis='y', right=True, labelright=False,
        direction='out', length=3.0, width=0.65, pad=2.0,
    )


def plot_grace_reconstruction(series: pd.DataFrame) -> Path:
    valid = series['has_grace_solution'] & series['grace_lwe_anom_cm'].notna()
    colors = {'grace': '#6FA9B8', 'grace_fill': '#EEF8F6', 'recon': '#7A5538'}

    fig, ax = plt.subplots(figsize=(11.5, 1.55), dpi=EXPORT_DPI)
    fig.subplots_adjust(left=0.105, right=0.900, top=0.91, bottom=0.32)
    style_timeseries_axis(ax)
    recon_ax = ax.twinx()
    recon_values = series['reconstruction_storage_proxy_anom_cm'].to_numpy(dtype=float)
    recon_valid = np.isfinite(recon_values)
    recon_x = mdates.date2num(series.loc[recon_valid, 'date'].to_numpy())
    smooth_x = np.linspace(recon_x.min(), recon_x.max(), len(recon_x) * 12)
    smooth_y = PchipInterpolator(recon_x, recon_values[recon_valid])(smooth_x)
    recon_line, = recon_ax.plot(
        mdates.num2date(smooth_x),
        smooth_y,
        color=colors['recon'],
        lw=1.5,
        label='Reconstruction',
        zorder=3,
    )
    grace_points = ax.scatter(
        series.loc[valid, 'date'],
        series.loc[valid, 'grace_lwe_anom_cm'],
        s=10,
        facecolor=colors['grace_fill'],
        edgecolor=colors['grace'],
        linewidth=1.5,
        label='GRACE',
        zorder=4,
    )
    ax.axhline(0, color='#666666', lw=0.45)
    ax.set_ylabel('GRACE\nanomaly (cm)', color=colors['grace'], labelpad=2)
    ax.set_ylim(-40, 40)
    ax.tick_params(axis='y', colors=colors['grace'], right=False)
    ax.spines['right'].set_visible(False)

    recon_ax.set_ylabel('Reconstruction\nanomaly (cm)', color=colors['recon'], labelpad=2)
    recon_ax.set_ylim(-20, 20)
    recon_ax.grid(False)
    recon_ax.tick_params(axis='x', top=False, bottom=False, labelbottom=False)
    recon_ax.tick_params(
        axis='y', colors=colors['recon'], direction='out', length=3.0, width=0.65, pad=2.0
    )
    for side in ['top', 'bottom', 'left']:
        recon_ax.spines[side].set_visible(False)
    recon_ax.spines['right'].set_visible(True)
    recon_ax.spines['right'].set_color(PANEL_SPINE_COLOR)
    recon_ax.spines['right'].set_linewidth(0.72)

    legend = ax.legend(
        [recon_line, grace_points],
        ['Reconstruction', 'GRACE'],
        frameon=True,
        ncol=2,
        loc='upper left',
        handlelength=1.55,
        columnspacing=0.95,
        borderpad=0.25,
        labelspacing=0.25,
        fontsize=7.5,
    )
    legend.get_frame().set_edgecolor('#B7B7B7')
    legend.get_frame().set_linewidth(0.55)
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_alpha(0.92)

    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.set_xlim(pd.Timestamp('2011-01-01'), pd.Timestamp('2023-12-31'))
    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha('right')

    out = OUT_DIR / 'Fig2_GRACE_reconstruction.png'
    fig.savefig(out, dpi=EXPORT_DPI, bbox_inches='tight', pad_inches=0.16)
    plt.close(fig)
    return out


In [ ]:
fig_paths = [plot_metric_map(spec) for spec in METRIC_SPECS]
fig_paths.extend(plot_metric_boxplot(spec) for spec in BOXPLOT_SPECS)
fig_paths.append(plot_class_map())

grace_series = compute_grace_reconstruction_series()
fig_paths.append(plot_grace_reconstruction(grace_series))

print('Fig2 outputs written to:')
for path in fig_paths:
    print(' ', display_path(path))


In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = None

for path in fig_paths:
    print(display_path(path))
    if Image is not None:
        display(Image(filename=str(path)))
